# LA Studio Direct Colab Voice Isolation GPU Worker

Choose **Runtime > Change runtime type > GPU**, then **Run all**. This starts a temporary Demucs GPU worker for vocals/background separation. It is an independent direct Colab route: it does not use, read, or proxy API Gateway credentials. Copy the final URL and token to **Voice Isolation > Direct Colab GPU** in LA Studio.

In [ ]:
import subprocess, sys

def run(*args):
    print('+', ' '.join(args))
    subprocess.run(args, check=True)

run('nvidia-smi')
run('ffprobe', '-version')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'demucs==4.0.1', 'fastapi>=0.115.0', 'uvicorn[standard]>=0.34.0', 'python-multipart>=0.0.20')


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_separation_worker.py')
WORKER.write_text(r'''
import os
import secrets
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

import torch
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import FileResponse

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before starting this worker.')

TOKEN = os.environ['LA_STUDIO_COLAB_TOKEN']
ROOT = Path('/content/la_studio_separation_jobs')
ROOT.mkdir(parents=True, exist_ok=True)
JOBS, JOB_LOCK = {}, threading.Lock()
MAX_UPLOAD_BYTES = 512 * 1024 * 1024
MAX_AUDIO_SECONDS = 30 * 60
ARTIFACT_TTL_SECONDS = 1800
ALLOWED_CONTENT_TYPES = {'audio/wav', 'audio/x-wav', 'audio/mpeg', 'audio/mp4', 'audio/webm', 'audio/ogg', 'audio/flac'}
JOB_SLOTS = threading.BoundedSemaphore(1)

def require_token(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

def media_duration_seconds(path: Path) -> float:
    probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=nokey=1:noprint_wrappers=1', str(path)], text=True, capture_output=True)
    try:
        duration = float(probe.stdout.strip())
    except ValueError:
        duration = 0.0
    if probe.returncode != 0 or duration <= 0.0:
        raise HTTPException(status_code=415, detail='audio is unsupported or could not be decoded')
    return duration

def update(job_id: str, **values) -> dict:
    with JOB_LOCK:
        job = dict(JOBS.get(job_id, {}))
        job.update(values)
        JOBS[job_id] = job
        return job

def cleanup(job_id: str) -> None:
    with JOB_LOCK:
        job = JOBS.pop(job_id, None)
    if job:
        shutil.rmtree(job.get('directory', ''), ignore_errors=True)

def run_job(job_id: str, directory: Path, source: Path) -> None:
    try:
        update(job_id, status='running', progress=25, detail='Demucs CUDA is separating vocals from background music')
        process = subprocess.Popen([sys.executable, '-m', 'demucs', '-n', 'htdemucs', '--two-stems', 'vocals', '--device', 'cuda', '--out', str(directory / 'out'), str(source)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        update(job_id, process=process)
        while process.poll() is None:
            with JOB_LOCK:
                cancelled = JOBS.get(job_id, {}).get('cancel_requested', False)
            if cancelled:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                update(job_id, status='cancelled', progress=0, detail='Separation cancelled')
                return
            time.sleep(0.5)
        stem_dir = directory / 'out' / 'htdemucs' / source.stem
        vocals, background = stem_dir / 'vocals.wav', stem_dir / 'no_vocals.wav'
        if process.returncode != 0 or not vocals.is_file() or not background.is_file():
            detail = (process.stdout.read() if process.stdout else '')[-2000:]
            update(job_id, status='failed', progress=0, detail='Demucs CUDA separation failed: ' + detail)
            return
        update(job_id, status='ready', progress=100, detail='Separated stems are ready', vocals=str(vocals), background=str(background))
    except Exception as error:
        update(job_id, status='failed', progress=0, detail=f'{type(error).__name__}: {error}')
    finally:
        threading.Timer(ARTIFACT_TTL_SECONDS, cleanup, args=[job_id]).start()
        JOB_SLOTS.release()

app = FastAPI(title='LA Studio Colab Separation Worker', docs_url=None, redoc_url=None, openapi_url=None)

@app.get('/health')
@app.get('/v1/health')
def health(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'status': 'ready', 'ready': True, 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'api_version': '1.0'}

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'contract_version': 1, 'capabilities': [{'id': 'voice-isolation', 'models': [{'id': 'htdemucs', 'stems': ['vocals', 'background'], 'formats': ['wav'], 'device': 'cuda'}]}]}

@app.post('/v1/audio/separations')
async def create_separation(file: UploadFile = File(...), stems: str = Form('vocals,background'), model: str = Form('htdemucs'), authorization: str | None = Header(default=None)):
    require_token(authorization)
    if model.strip().lower() != 'htdemucs' or stems != 'vocals,background':
        raise HTTPException(status_code=422, detail='this worker supports htdemucs vocals,background only')
    if file.content_type not in ALLOWED_CONTENT_TYPES:
        raise HTTPException(status_code=415, detail='unsupported audio MIME type')
    suffix = Path(file.filename or 'source.wav').suffix.lower() or '.wav'
    if suffix not in {'.wav', '.mp3', '.m4a', '.mp4', '.webm', '.ogg', '.flac'}:
        raise HTTPException(status_code=415, detail='unsupported audio filename extension')
    if not JOB_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='the Colab separation worker is busy; retry shortly')
    job_id = secrets.token_urlsafe(18)
    directory = ROOT / job_id
    directory.mkdir(parents=True, exist_ok=True)
    source = directory / ('source' + suffix)
    try:
        with source.open('wb') as output:
            while chunk := await file.read(1024 * 1024):
                output.write(chunk)
                if output.tell() > MAX_UPLOAD_BYTES:
                    raise HTTPException(status_code=413, detail='audio exceeds 512 MB upload limit')
        if source.stat().st_size <= 0:
            raise HTTPException(status_code=413, detail='audio must not be empty')
        if media_duration_seconds(source) > MAX_AUDIO_SECONDS:
            raise HTTPException(status_code=413, detail='audio exceeds the 30 minute duration limit')
    except Exception:
        shutil.rmtree(directory, ignore_errors=True)
        JOB_SLOTS.release()
        raise
    finally:
        await file.close()
    update(job_id, status='queued', progress=10, detail='Audio uploaded; Demucs CUDA job is queued', directory=str(directory), cancel_requested=False)
    threading.Thread(target=run_job, args=(job_id, directory, source), daemon=True).start()
    return {'job_id': job_id, 'status': 'queued', 'progress': 10}

@app.get('/v1/audio/separations/{job_id}')
def separation_status(job_id: str, authorization: str | None = Header(default=None)):
    require_token(authorization)
    with JOB_LOCK:
        job = dict(JOBS.get(job_id, {}))
    if not job:
        raise HTTPException(status_code=404, detail='separation job not found')
    return {key: job.get(key) for key in ('status', 'progress', 'detail') if key in job} | {'job_id': job_id}

@app.get('/v1/audio/separations/{job_id}/artifacts/{stem}')
def artifact(job_id: str, stem: str, authorization: str | None = Header(default=None)):
    require_token(authorization)
    if stem not in {'vocals', 'background'}:
        raise HTTPException(status_code=404, detail='unknown stem')
    with JOB_LOCK:
        job = dict(JOBS.get(job_id, {}))
    path = Path(job.get(stem, ''))
    if job.get('status') != 'ready' or not path.is_file():
        raise HTTPException(status_code=409, detail='stem is not ready')
    return FileResponse(path, media_type='audio/wav', filename=stem + '.wav')

@app.delete('/v1/audio/separations/{job_id}')
def cancel_separation(job_id: str, authorization: str | None = Header(default=None)):
    require_token(authorization)
    with JOB_LOCK:
        if job_id not in JOBS:
            raise HTTPException(status_code=404, detail='separation job not found')
        JOBS[job_id]['cancel_requested'] = True
        JOBS[job_id]['status'] = 'cancelling'
    return {'job_id': job_id, 'status': 'cancelling'}
''')
print('Worker source written:', WORKER)


In [ ]:
import os, re, secrets, subprocess, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_TOKEN'] = TOKEN
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_separation_worker:app', '--host', '127.0.0.1', '--port', '3924'], cwd='/content', env=env)
for _ in range(45):
    try:
        request = urllib.request.Request('http://127.0.0.1:3924/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('LA Studio separation worker did not become ready')
subprocess.run(['bash', '-lc', 'wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3924', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate(); tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')
print('\nLA_STUDIO_COLAB_SEPARATION_URL=' + public_url)
print('LA_STUDIO_COLAB_SEPARATION_TOKEN=' + TOKEN)
print('MODEL=htdemucs; STEMS=vocals,background')
